# 模型上下文协议

## 问题描述

2025年以前，每个host（运行一个LLM的东西）一个每个server（暴露工具和数据的东西）都有定制的协议，这意味着你需要维护NxM的继承矩阵。

MCP将其拍平了。一个server暴露工具、资源、提示词模板。任何兼容的host都能发现并使用他们，而不需要自定义胶水层。

## 基本概念

MCP：一个host，一个server，三种能力

- 工具。   模型可以调用的函数
- 资源。   模型或者用户可以请求访问的只读上下文。
- 提示词。 可复用的提示词模板。

host 是LLM应用，比如（Claude客户端），client是host的一个子组件，只与一个server通话。server就是你的代码。一个host可以同时挂载多个server。

### MCP不是什么

- 不是检索API。   RAG还是决定召回什么，MCP将召回结果暴露出来作为资源。
- 不是Agent框架。  MCP是管道，框架在此之上。
- 不与Anthropic绑定。

# 原理代码

In [1]:
from dataclasses import dataclass
from typing import Any, Callable
import queue


@dataclass
class Tool:
    name: str
    description: str
    input_schema: dict[str, Any]
    handler: Callable[..., Any]
    destructive: bool = False

@dataclass
class Resource:
    uri: str
    description: str
    handler: Callable[[], str]

@dataclass
class Prompt:
    name: str
    description: str
    arguments: list[str]
    handler: Callable[..., Any]

class MCPServer:
    def __init__(self, name:str):
        self.name = name
        self.tools : dict[str, Tool] = {}
        self.resources : dict[str, Resource] = {}
        self.prompts : dict[str, Prompt] = {}

    def tool(
        self,
        name: str,
        description: str,
        schema: dict[str, Any],
        *,
        destructive: bool = False,
    ):
        def decorator(
            fn: Callable[..., Any]
        ) -> Callable[..., Any]:
            self.tools[name] = Tool(
                name, description, schema, fn, destructive
            )
            return fn
        return decorator

    def resource(
        self,
        uri: str,
        description: str,
    ):
        def decorator(
            fn: Callable[[], str]
        ) -> Callable[[], str]:
            self.resources[uri] = Resource(uri, description, fn)
            return fn
        
    def prompt(
        self,
        name: str,
        description: str,
        arguments: list[str],
    ):
        def decorator(
            fn: Callable[..., Any]
        ) -> Callable[..., Any]:
            self.prompts[name] = Prompt(name, description, arguments, fn)
            return fn
        return decorator

    def handle(self, message):
        # Dispatch by your-self...
        pass

class MCPClient:
    def __init__(self, server: MCPServer):
        self.server = server
        self._id = 0
        self.inbox: queue.SimpleQueue[dict[str, Any]] = queue.SimpleQueue()

    def _next_id(self):
        self._id += 1
        return self._id
    
    def request(
        self,
        method,
        params
    ):
        # Transmit the request to the server
        response = self.server.handle(
            {
                "id": self._next_id(),
                "method": method,
            }
        )
        return response

# 商业API

In [ ]:
# 使用独立包 fastmcp（uv/pip: fastmcp）。
# mcp.server.fastmcp 在 mcp 2.x 下会因 LifespanContextT 导入失败。
from fastmcp import FastMCP

mcp = FastMCP("demo-server")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two integers"""
    return a + b

@mcp.resource("config://app")
def app_config() -> str:
    """The application JSON configuration"""
    return '{"env": "prod", "region":"us-east-1"}'
    
@mcp.prompt()
def code_review(language: str, code: str) -> str:
    """Review the code in the given language"""
    return f"Reviewing {language} code:\n{code}"


RuntimeError: Already running asyncio in this thread

In [ ]:
from fastmcp import FastMCP
from mcp import ClientSession, StdioServerParameters

params = StdioServerParameters()
